PROMPT TUNING E ADAPTERS (LoRA, PEFT)

Gli LLM sono potenti, ma hanno anche miliardi di paramtri.
Come possiamo addestrare questo gigante?
con i PEFT è possibile fare addestramento su modelli giganti usanto la potenza di un comune pc da casa.

- Fine-Tunning efficiente: superare il limiti dei 70 milioni di parametri
- LoRA: iniette matrici di rango basso nell'attention
- Vantaggi e Portabilità: adapter da pochi MB contro giganti da centinaia di GB

La fine dell'era del addestramento totale.

L'ERA dei Parametri Congelati
Addestrare un intero modello da 70B di parametri richiede una potenza di calcolo è una memoria VRAM inaccessibili per la maggior parte del pc.
La soluzione risiede nel Parameter-Efficient Fine-Tunning PEFT dove manteniamo i pesi originali del modello 'congelato' e addestriamo solo una minima frazion di nuovi parametri.
Invece di svegliare tutti i neroni del gigante, li congeliamo, il cervello originale rimane intatto e noi aggiungiamo solo un piccolo modulo esterno

Ho un LLM già addestrato con miliardi di parametri. Come lo specializzo sui miei dati senza riaddestrarlo completamente?

La famiglia di tecniche si chiama PEFT (Parameter-Efficient Fine-Tuning). 
L'idea è congelare quasi tutto il modello e addestrare solo una piccola quantità di parametri aggiuntivi. 
Hugging-Face considera LoRA, IA, AdaLoRA, Prompt Tuning, Prefix Tuning e altre tecniche all'interno di questa famiglia.

Il Full Fine-Tuning è potente ma costoso.
Il modello, già di per se, ha miliardi di parametri (modello base 7 miliardi di parametri). Con il Full Fine-Tuning dovrei cambiare miliardi di parametri, processo potente ma costoso. Avrei bisogno di molta molta memoria perchè non devo conservare solo i pesi, ma anche i gradienti e stato dell'optimizer (durante il training).
Dataset aziendale -> LLM -> backpropagation -> modifico praticamente tutti i pesi
Inoltre, se ho la necessità di avere 3 specializzazioni (esempio modello commerciale, modello tecnico e modello amministrativo) richiede di mantenere tutti i modelli completi in memoria.

PEFT cerca di evitare questo.
Filosofia di PEFT:
modello base (70 miliardi di parametri) -> lo congelo -> estraggo pochi parametri addestrabili -> specializzazione.
Invece di modificare 7.000.000.000 parametri, posso addestrare qualche milione di parametro (o anche meno).
Questo riduce la memoria, il costo computazionale e dimensionale dei checkpoint da salvare.
Permette di spacializzare il modello agendo solo su una frazione infinitesimale dei pesi totali.
Non serve solo spazio per i pesi, ma anche per i gradienti e lo stato degli optimizer.

Tornando all'esempio della specializzazione di prima non parto più dal modello completo, ma dal modello base
modello base + adapter commerciale 
modello base + adapter tecnico
modello base + adapter amministrativo
Non devo dupliare (triplicare) tutto il modello.

Ma come possiamo iniettare questa nuova conoscenza senza toccare i pesi originali?
esistono diverse scuole di pensiero per l'adattamento:
- Prompt tuning: consiste nell'aggiungere vettori addestrabili all'inizio del prompt, lasciando il modello completamente intatto
- Prefix tuning: simile al prompt tuning ma inietta i parametri addestrabili in ogni layer del Transformer per un controllo più profondo.
- Adapter Layer: inserimento di piccoli-blocchi Feed-Forward tra i layer esistenti per catturare la logica del nuovo task. 
Ognuno di questi metodi ha l'obbbiettivo di lasciare la spina dorsale del modello congelato
Stiamo parlando di addestrare meno dello 0.1% dei parametri totali.

Differenza tra Prompt Engineering e Prompt Tunning

Prompt Enginnering
Quando scrivi un prompt, esempio "sei un esperto ERP estrai dall'ordine: codice artcolo, quantità e prezzo. Restituisci JSON" stai facendo prompt engineering, non addestri nulla. Non esiste backpropagation e non vengono modificati i pesi.
Prompt scritto da me -> LLM -> risposta
Domani cambi, manualmente, prompt, il comportamento cambia.

Prompt Tunning
Con il prompt tunint il prompt non è costruito soltanto da parole scritte dame. Vengono creati dei soft prompt, cioè vettori addestrabili.
Supponiano che il prompt diventi:
[token virtuale 1]
[token virtuale 2]
[token virtuale 3]
...
[token virtuale 20]
"Classifica questa mail..."
Questi token virtuali non corrispondono, necessariamente, a parole leggibili
Sono vettori
Esempio
[token virtuale 1]=[0.18,-0.72,0.51,...]
Durante il training:
LLM congelato - soft prompt -> addestrato
Huggin-Face definisce il Prompt Tunning come l'aggiunta di token virtuali apprendibili nello spazio degli embedding mentre i parametri del modello pretrained restano congelati.
Esempio
Supponiamo di dover classificare una mail in: ordine, offerta, reclamo, altro
Prompt Engineering: "Classifica questa mail come: ordine, offerta, reclamo, altro"
Prompt Tunning: [V1][V2][V3][V4][V5] "Classifica questa mail..."
dove V1...V5 vengon imparati dal training
L'algoritmo cerca automaticamente quali embedding iniziali aiutano maggiormente il modello a svolgere quel compito.
Il prompt nel "prompt engineering" non è scritto a mano ma appreso

Prefix Tunning
Nel prompt tunning i token virtuali vengono aggiunti all'input:
[token virtuale 1]
[token virtuale 2]
[token virtuale 3]
...
[token virtuale 20]
"Classifica questa mail..."
Nel Prefix Tunning vengono introdotte rappresentazioni apprendibili che influenzano più direttamente i vari layer del Transformer.
prefix -> transformer layer 1
prefix -> transformer layer 2
prefix -> transformer layer 3
E' più invasivo del Prompt Tunning, pur lasciando congelato il moello base.

Adapters
L'idea è:
transformer originale -> piccolo modulo traingbile -> transformer originale -> piccolo modulo traingbile
Il modello originale rimane congelato
Gli adapter introducono piccole componenti che imparano la specializzazione
Esempio: 
Modello Generico
"conosco italiano, inglese, programmazione, storia, matematica, ..."
+
Adapter ERP

LoRA
LoRA è un tipo particolare di adattamento
LoRA = Low-Rank Adaptation
Per specializzare un modello enorme probabilmente non è necessario modificare liberamente ogni elemento delle grandi matrici dei pesi.
Un LLM conosce già moltissimo
Se vuoi insegnarli:
"come interpretare gli ordini cliente della mia azienda"
non devo insegnarli nuovamento:
italiano, matematica, grammatica, cos'è un ordine, cos'è un prezzo, ecc
Devi spostare la direzione del modello in una direzione relativamente specifica.
LoRA cerca di rappresentare questa correzione in poche dimensioni.
Il parametro r (rankg) di LoRA identifica i parametri necessari: 
R basso: meno parametri, meno memoria, meno capacità di adattamente
R alto: più parametri, più capacità, più memoria
Non esiste un valore di R migliore, dipende dal modello, dai dati e dal task

LoRA ipotizza che gli aggiornamenti dei pesi durante il fine-tuning abbiano un 'rango intrinseco' basso, ovvero non servano tutte le dimensioni della matrice per imparare un nuovo compito.
Invece di aggiornare la matrice dei pesi originale, LoRA addestra due matrici molto più piccole la cui scomposizione approssima l'aggiornamento desiderato.

Scomposizione delle Merici
- La matrice originale W rimane congelata (bloccata) e non riceve alcun aggiornameto durante il training.
- Matrice Low-Rank: usiamo due matrici A e B, di rango r, dove r è molto più piccolo della dimensione originale. Se W è una griglia enorme, es 4096x4096 A e B sono strisce sottili, esempio 4096x8. Il risparmio è enorme
- Somma degli output:l'output finale del layer è la somma del passaggio attaverso W e del passaggio attraverso il prodotto di B e A
- Durante l'inferenza, le matrice LoRA possono essere fuse con W, eliminando qualsiasi latenzza aggiuntiva.
Durante il calcolo l'input passa attarverso il gigante congelato W e il prodotto di B e A, alla fine abbiamo il risultato.
In inferenza possiamo anche fondere matematicamente queste matrice per non aver alcun rallentamento.

Dinamiche di Training
- Inizzializzazione: la matrice A viene inizializzata in modo cauale (gaussiana), mentre B viene inizializzata a zero, questo perchè vogliamo che l'output, al primo passo, sia esattamente uguale a quello del modello originale.
Iniziamo quindi dal punto 0 e lasciamo che la rete scopra solo le deviazioni necessarie.
- Scelta del Rango: solitamente un rango r tra 4 e 64 è sufficiente per compiti complessi riducendo i parametri di migliaia di volte. Di solito r è poccolo o 8 o 16.
- Alpha Scaling: un parametro di scala che permette di pesare l'influenza dell'adapter senza dover cambiare il learning rate.

Perchè una cosa così piccola riesce ad influenzare un gigante così grande?
Funziona perchè il linguaggio ha una strttura intrinseca molto ridondante.
Il cambiamente semantico necessario per un nuovo task non richiede di stravolgere la struttura del linguaggio, ma solo di orientarla.
Non stiamo insegnando al modello come pensare da zero.
Questa intuizione permette a LoRA di catturare la delta dell'apprendimento con numero ridicolo di variabili.

Portabilità e il futuro degli adapter
Uno dei vantaggi più sottovalutati di LoRA è che i pesi addestrati occupano pochissimo spazio, spesso nell'ordine dei 10-100 megabyte.
Questo permette di distribuire adapter specializzati in modo rapido e di scambiarli al volo su un unico modello base caricato in memoria.

Scitching e Versatilità
Gestire molteplici task simultaneamente.
- Multi-tasking: caricando un solo modello da 140GB possiamo applicare diversi adapter da 50MB per rispondere a esigenze diverse.
- Versionamento: gli adapter facilitano il test di diverse iterazioni di addestramento senza dover gestire file giganti.
- Collaborazione: la community può condividere 'adapter' specifici su piattaforme come Hugging Face senza violare licenze di modello chiusi.
- L'uso degli adapter trasforma il modello base in un processore universale su cui montare schede di espansione specializzate.

Ma possiamo spingerci ancora oltre riducendo ancora di più lo spazio.

Deployment Strategico
Zero Latency Merge: Fondere l'adapter nel modello base permete di avere le performance del fine-tuning senza costi aggiuntivi di calcolo
Quantizzazione (QLoRA): combinare la quantizzazione a 4-bit (che comprime il peso del modello di 8 volte) con LoRA permette di addestrare modelli da 70B parametri su una singola scheda video professionale da 48 GB
Il modello base è una roccia congelata e compressa, mentre LoRA è lo strato sottile ad alta precisione che impara.
Inference Streaming: servire diversi utenti con adapter diversi sullo stesso hardware è la chiave per la scalabilità dei servizi AI.

Impatto Economico di questa scelta
Passare da 100GB a 100MB per ogni variante del modello, riduce i costi di storage e i tempi di caricamento di tra ordini di grandezza. Spazio su disco e velocità di caricamento ne beneficiano.
Questa modularità è ciò che rende possibile l'integrazione di LLM personalizzati in applicazioni mobile o edge computing.

In [1]:
import torch
from diffusers import DiffusionPipeline

# Configurazione dell'hardware: viene data priorità alla GPU (CUDA) se disponibile.
# L'uso di una GPU è fondamentale per garantire tempi di generazione accettabili con modelli complessi come SDXL.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Identificativo del modello base di Stable Diffusion XL 1.0.
model_id = "stabilityai/stable-diffusion-xl-base-1.0"

# --- 1. CARICAMENTO DEL MODELLO BASE ---
# La classe DiffusionPipeline gestisce l'intero workflow di generazione (modello UNet, VAE, text encoders).
# Utilizziamo torch.float16 (precisione dimezzata) per ridurre drasticamente il consumo di memoria video (VRAM)
# senza compromettere visibilmente la qualità dell'immagine finale.
pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)

# Spostiamo l'intera pipeline sulla memoria della GPU (o CPU) per l'esecuzione.
pipe.to(device)

# Definizione del prompt: la descrizione testuale che guida il processo di diffusione.
prompt = "A futuristic city with flying cars, sunset, cinematic lighting"

print("--- GENERAZIONE CON MODELLO BASE ---")
# Il metodo __call__ della pipeline avvia il processo di denoising iterativo.
# num_inference_steps definisce quanti passaggi di campionamento verranno effettuati (valore tipico per SDXL è tra 30 e 50).
# .images[0] estrae la prima immagine generata dalla lista dei risultati.
image_base = pipe(prompt, num_inference_steps=30).images[0]

# Salvataggio dell'immagine prodotta dal modello originale sul disco.
image_base.save("city_base.png")

# --- 2. INTEGRAZIONE DEL MODULO LORA (Low-Rank Adaptation) ---
# LoRA permette di modificare il comportamento del modello iniettando piccoli pesi addizionali (matrici A e B)
# negli strati di attenzione (Attention Layers), senza dover ricaricare l'intero modello pesante.
# Questo esempio carica uno stile specifico (Pixel Art).
lora_model_id = "nerjs/pixel-art-xl"

# Il metodo load_lora_weights scarica i pesi LoRA e li applica dinamicamente alla pipeline esistente.
# Questa operazione è molto più veloce rispetto al caricamento di un nuovo modello completo.
pipe.load_lora_weights(lora_model_id)

print("\n--- GENERAZIONE CON LORA ATTIVO ---")
# Eseguiamo nuovamente la generazione con lo stesso prompt. 
# La pipeline ora utilizzerà i pesi base "corretti" dai pesi LoRA, applicando lo stile pixel art al contenuto.
image_lora = pipe(prompt, num_inference_steps=30).images[0]

# Salvataggio della versione stilizzata con LoRA.
image_lora.save("city_pixel_art.png")

# --- 3. GESTIONE DELLA MODULARITA ---
# Una delle caratteristiche principali di questo approccio è la possibilità di scaricare i pesi aggiuntivi.
# Il metodo unload_lora_weights rimuove i pesi applicati in precedenza, ripristinando lo stato originale del modello.
# Questo permette di passare da uno stile all'altro mantenendo lo stesso oggetto pipeline in memoria.
pipe.unload_lora_weights()

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\barbara\.cache\huggingface\hub\models--stabilityai--stable-diffusion-xl-base-1.0. Caching files will still work but in a degraded version that might require more space on y

--- GENERAZIONE CON MODELLO BASE ---


100%|██████████| 30/30 [103:18:17<00:00, 12396.58s/it]  
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\diffusers\pipelines\stable_diffusion_xl\pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


ValueError: PEFT backend is required for `load_lora_weights()`.